# FastOTF2 Converter — Scaling & Performance Notebook

A single notebook that **collects** converter timings across traces and node counts, then
**graphs** strong- and weak-scaling behaviour. It replaces `converter-scaling.ipynb` and
`perf_matrix.ipynb`.

**Workflow**
1. *Setup* (§0): download the container, configure the `e4s-cl` environment, create a
   timestamped run folder.
2. *Inventory* (§1): measure each trace's real size — this drives the weak-scaling math.
3. *Design* (§2): one cell decides **what** to run — `STRONG`, `WEAK`, or `ALL`. Set
   `DRY_RUN = True` first to sanity-check submission and graphs with a tiny, fast matrix.
4. *Submit* (§3): dependency-chained Slurm jobs; timings captured via `--timings-csv`.
5. *Analyse* (§5+): load the CSVs and draw graphs with **plotnine**. This half is
   self-contained — to study a **previous** run (no new data), restart the kernel, run the
   §5 config cell with `ANALYZE_RUN` set, then run the graph cells.

**Conventions**
- **`traced_nodes`** = the node count a trace was *captured* on (its size label);
  **`nl`** = the converter launch node count (`-nl`).
- Sizes are **GiB (1024³)**, measured with `du -sb` (fixes the old `du -h` / `humanfriendly`
  decimal mismatch).
- **Strong** scaling fixes a trace and varies `nl`. **Weak** scaling gives each trace a
  *bespoke* `nl = round(size / target)` so data-per-node stays ≈ constant.

# 0. Environment, container & e4s-cl setup

Prepares everything a job needs: Python imports, user paths, a **timestamped run folder**
(so runs never clobber each other), the OFI container (pulled/converted to a `.sif`), and
the `e4s-cl` profile (binds host paths + sources the OTF2 pre-exec script). Each run of this
section creates a fresh run folder; to **analyse a previous run** without collecting new
data, skip straight to §5 and set `ANALYZE_RUN` there.

**`SYSTEM`** picks which cluster's settings to use (`"frontier"` or `"other-ex"`) — cpus per
task, container URL, apptainer PATH/cache setup, `sbatch` extras (account/mail), and trace
paths (§1) all follow from this one switch via `SYSTEM_CONFIGS`. The Slurm **account** and
**mail** address are secrets and are never hardcoded here — create a gitignored
`workflows/scaling/local_secrets.py` with `SLURM_ACCOUNT` / `SLURM_MAIL_USER` (env vars work
too, but only if set before the Jupyter server process itself starts — an already-running
kernel won't pick up a shell `export` after the fact).

In [ ]:
# --- Imports ---
import os
import re
import json
import subprocess
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from plotnine import *  # noqa: F401,F403

import workflows

# --- Which system are we running on? ---------------------------------------
# Flip this one line to switch clusters; everything system-specific lives in
# SYSTEM_CONFIGS below so nothing else in the notebook needs to change.
SYSTEM = "frontier"   # "frontier" | "other-ex"

# --- Secrets (never hardcode these in the notebook) -------------------------
# Preferred: a gitignored local_secrets.py next to this notebook (workflows/scaling/),
# since env vars exported in a terminal are NOT inherited by an already-running Jupyter
# kernel/server -- exporting them and restarting the kernel does nothing, you'd need to
# fully relaunch the process that started Jupyter itself. local_secrets.py sidesteps that:
#   SLURM_ACCOUNT = "csc688"
#   SLURM_MAIL_USER = "you@example.com"
# Falls back to SLURM_ACCOUNT / SLURM_MAIL_USER env vars if that file doesn't exist.
try:
    from local_secrets import SLURM_ACCOUNT, SLURM_MAIL_USER
except ImportError:
    SLURM_ACCOUNT   = os.environ.get("SLURM_ACCOUNT")
    SLURM_MAIL_USER = os.environ.get("SLURM_MAIL_USER")


def _mail_sbatch_args():
    return (["--mail-type=BEGIN,END,FAIL", f"--mail-user={SLURM_MAIL_USER}"]
            if SLURM_MAIL_USER else [])


def _account_sbatch_args():
    return [f"--account={SLURM_ACCOUNT}"] if SLURM_ACCOUNT else []


# --- Per-system configuration ------------------------------------------------
# go_dir / apptainer_bin_path / apptainer_cache_dir: set to None when a system
# doesn't need them (e.g. Frontier already has apptainer on PATH).
SYSTEM_CONFIGS = {
    "frontier": {
        "thread_count":        56,   # --cpus-per-task per job
        "go_dir":              None,
        "apptainer_bin_path":  None,
        "apptainer_cache_dir": None,
        "container_url": "ghcr.io/hpc-ai-adv-dev/fastotf2/fastotf2-converter-libfabric2.3.1:latest",
        # Frontier's libfabric (1.22.0) is much older than the other system's (2.3.1).
        # The 2.3.1-linked container is expected to work, but if it doesn't, switch to:
        #   "ghcr.io/hpc-ai-adv-dev/fastotf2/fastotf2-converter-frontier:latest"
        # apptainer can pull OCI images directly -- no need for podman + convert-to-sif.sh.
        "container_pull_method": "apptainer",
        "sbatch_extra_args":   _account_sbatch_args() + _mail_sbatch_args(),
        "trace_inputs": {
            2:   "/lustre/orion/csc688/world-shared/scorep-amd/runs/scratch/4098287/frontier-2-node-single-HPL-run",
            4:   "/lustre/orion/csc688/world-shared/scorep-amd/runs/scratch/4098294/frontier-4-node-single-HPL-run",
            8:   "/lustre/orion/csc688/world-shared/scorep-amd/runs/scratch/4098296/frontier-8-node-single-HPL-run",
            16:  "/lustre/orion/csc688/world-shared/scorep-amd/runs/frontier-16-node-single-HPL-run",
            32:  "/lustre/orion/csc688/world-shared/scorep-amd/runs/frontier-32-node-single-HPL-run",
            128: "/lustre/orion/csc688/world-shared/scorep-amd/runs/scorep-rocHPL/experiments/one-cabinet",
            384: "/lustre/orion/csc688/world-shared/scorep-amd/runs/scorep-rocHPL/experiments/three-cabinet",
        },
    },
    "other-ex": {
        "thread_count":        64,
        "go_dir":              "/lus/scratch/crickett/go/",
        "apptainer_bin_path":  "/lus/scratch/crickett/software/usr/bin",
        "apptainer_cache_dir": "/lus/bnchlu1/khandeka/.local/apptainer/",
        "container_url": "ghcr.io/hpc-ai-adv-dev/fastotf2/fastotf2-converter-libfabric2.3.1:latest",
        # apptainer can pull OCI images directly -- no need for podman + convert-to-sif.sh.
        # (If this system's apptainer can't reach the registry directly, switch this back to
        # "podman", which converts via podman pull + convert-to-sif.sh instead.)
        "container_pull_method": "apptainer",
        "sbatch_extra_args":   [],
        "trace_inputs": {
            2:   "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-2-node-single-HPL-run",
            4:   "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-4-node-single-HPL-run",
            8:   "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-8-node-single-HPL-run",
            16:  "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-16-node-single-HPL-run",
            32:  "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-32-node-single-HPL-run",
            128: "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-128-node-single-HPL-run",
            384: "/lus/bnchlu1/adt/otf2-traces/frontier/frontier-384-node-single-HPL-run",
        },
    },
}

_cfg = SYSTEM_CONFIGS[SYSTEM]
if SYSTEM == "frontier" and not (SLURM_ACCOUNT and SLURM_MAIL_USER):
    raise RuntimeError(
        "SYSTEM='frontier' needs SLURM_ACCOUNT and SLURM_MAIL_USER -- create "
        "workflows/scaling/local_secrets.py (gitignored) with those two variables, or set "
        "them as environment variables before the Jupyter server process itself is started."
    )
go_dir              = _cfg["go_dir"]
apptainer_bin_path  = _cfg["apptainer_bin_path"]
apptainer_cache_dir = _cfg["apptainer_cache_dir"]
container_url       = _cfg["container_url"]
CONTAINER_PULL_METHOD = _cfg["container_pull_method"]
SBATCH_EXTRA_ARGS   = _cfg["sbatch_extra_args"]
entrypoint_path = "/workspace/fastotf2/apps/FastOTF2Converter/target/release/FastOTF2Converter_real"

STRATEGY     = "locgroup_dist_block"   # multi-locale distribution strategy
FORMAT       = "PARQUET"               # converter output format
THREAD_COUNT = _cfg["thread_count"]     # --cpus-per-task per job

# --- Run folder & logging ---
WORKFLOW_DIR = Path.cwd()               # the workflows/scaling directory

# Every execution of this cell starts a NEW, timestamped run folder — it never clobbers a
# previous run. (To *analyse* an existing run without collecting new data, don't touch this;
# jump to §5 and set ANALYZE_RUN there.)
RUN_TAG = f"run_{datetime.now():%Y%m%d_%H%M%S}"

RUN_DIR            = WORKFLOW_DIR / "out" / RUN_TAG
TIMINGS_DIR        = RUN_DIR / "timings"         # full-sweep --timings-csv output (tiny)
SAMPLE_TIMINGS_DIR = RUN_DIR / "timings_sample"  # SAMPLE_RUN timings live here instead, so a
                                                  # quick sample can never mix with a full sweep
SLURM_LOG_DIR = RUN_DIR / "slurm_logs"   # sbatch job stdout/stderr
RUN_LOG_DIR   = RUN_DIR / "run_logs"     # per-run scripts + converter.log
OUTPUT_ROOT   = RUN_DIR / "pq"           # converter Parquet output (large; safe to delete)
PLOTS_DIR     = RUN_DIR / "plots"        # saved figures (optional)

for d in (RUN_DIR, TIMINGS_DIR, SAMPLE_TIMINGS_DIR, SLURM_LOG_DIR, RUN_LOG_DIR,
          OUTPUT_ROOT, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

workflows.set_workflow_log(str(RUN_DIR / "workflow.log"))  # note: this chdir's to RUN_DIR ...
os.chdir(WORKFLOW_DIR)                                      # ... so restore the workflow dir

print(f"SYSTEM  : {SYSTEM}")
print(f"RUN_TAG : {RUN_TAG}")
print(f"RUN_DIR : {RUN_DIR}")
print(f"timings : {TIMINGS_DIR}  (sample timings: {SAMPLE_TIMINGS_DIR})")


In [ ]:
# Environment variables for apptainer / e4s-cl.
# go_dir / apptainer_bin_path / apptainer_cache_dir are None on systems (e.g. Frontier)
# that already have apptainer on PATH -- skip the corresponding env var entirely there.
if apptainer_cache_dir:
    os.environ["APPTAINER_CACHEDIR"] = apptainer_cache_dir

_path_additions = []
if go_dir:
    _path_additions.append(os.path.join(go_dir, "bin"))
if apptainer_bin_path:
    _path_additions.append(apptainer_bin_path)
if _path_additions:
    os.environ["PATH"] = os.environ.get("PATH", "") + os.pathsep + os.pathsep.join(_path_additions)

if go_dir:
    os.environ["LD_LIBRARY_PATH"] = (os.environ.get("LD_LIBRARY_PATH", "") + os.pathsep
                                     + os.path.join(go_dir, "lib"))

# Download (or reuse) the container as a .sif in the workflow directory. Method depends on
# SYSTEM: "apptainer" pulls the OCI image directly (no podman needed -- e.g. Frontier);
# "podman" converts via podman + convert-to-sif.sh (this system's existing convention).
workflows.print_and_log(f"Fetching container ({CONTAINER_PULL_METHOD}): {container_url}")
container_name = workflows.download_custom_container(container_url, method=CONTAINER_PULL_METHOD)
container_path = str(WORKFLOW_DIR / container_name)
print(f"Container: {container_path}")

# Pre-exec script sourced inside the container (OTF2 libs + Chapel comm heap).
# CHPL_RT_MAX_HEAP_SIZE caps the fixed communication heap the NIC must pin per node;
# 70% was too large to register on the Slingshot/CXI NIC (multi-locale jobs died at
# init), so we use 50%.
preexec_path = WORKFLOW_DIR / "e4s-cl-preexec.sh"
preexec_path.write_text(
    "#!/bin/bash\n"
    'export LD_LIBRARY_PATH="/opt/otf2/lib:${LD_LIBRARY_PATH}"\n'
    "export CHPL_RT_MAX_HEAP_SIZE=50%\n"
)

# Register host paths with the e4s-cl profile so the container can read/write them.
# WORKFLOW_DIR contains RUN_DIR, so the run's outputs/timings are covered here;
# trace directories (elsewhere on the filesystem) are added in §1.
workflows.run_cmd(f"e4s-cl profile edit --add-files {WORKFLOW_DIR}")
workflows.run_cmd(f"e4s-cl profile edit --source {preexec_path}")
print("e4s-cl profile updated (workflow dir + pre-exec).")

# 1. Trace inventory & sizes

Measures each trace's real on-disk size with `du -sb` (bytes → GiB) and caches it to
`trace_sizes.json` (sizes are fixed, so this runs once). These sizes drive the
weak-scaling math in §2. Each trace directory is also registered with the `e4s-cl` profile
so the container can read it.

In [ ]:
# Traces keyed by the node count they were CAPTURED on (the "size" label).
# Sourced from SYSTEM_CONFIGS[SYSTEM]["trace_inputs"] so switching SYSTEM in §0 also
# switches which trace paths get used here.
TRACE_INPUTS = _cfg["trace_inputs"]
TRACE_FILE = "traces.otf2"            # file within each trace directory
_GIB = 1024 ** 3


def measure_trace_sizes_gib(trace_inputs, cache_path):
    """du -sb each trace dir -> GiB, cached to JSON (trace sizes are fixed).

    Uncached traces print progress + elapsed time -- du -sb has to stat() every file under
    each trace's traces/ subdir (tens of thousands for the largest traces), which is slow
    on Lustre and would otherwise look hung with no output for several minutes."""
    cache_path = Path(cache_path)
    cache = json.loads(cache_path.read_text()) if cache_path.exists() else {}
    sizes = {}
    for traced_nodes, path in trace_inputs.items():
        key = str(traced_nodes)
        if key not in cache:
            print(f"  measuring size of trace s{traced_nodes} ({path}) -- "
                  f"du -sb over many files on Lustre, may take a while...")
            t0 = time.monotonic()
            out = subprocess.run(["du", "-sb", path], capture_output=True, text=True, check=True)
            cache[key] = int(out.stdout.split()[0]) / _GIB
            print(f"    -> {cache[key]:.2f} GiB ({time.monotonic() - t0:.0f}s)")
        sizes[traced_nodes] = cache[key]
    cache_path.write_text(json.dumps(cache, indent=2, sort_keys=True))
    return sizes


trace_sizes_gib = measure_trace_sizes_gib(TRACE_INPUTS, RUN_DIR / "trace_sizes.json")

# Register trace dirs with the e4s-cl profile (so the container can read them).
for path in TRACE_INPUTS.values():
    workflows.run_cmd(f"e4s-cl profile edit --add-files {path}")

trace_table = pd.DataFrame(
    [(t, round(trace_sizes_gib[t], 3), TRACE_INPUTS[t]) for t in sorted(TRACE_INPUTS)],
    columns=["traced_nodes", "size_GiB", "path"],
)
display(trace_table)

# 2. Experiment design — what to run

Two independent switches at the top of the cell:

- **`SAMPLE_RUN`** — a small but *real* matrix (4 small traces, node grid `1–16`, 2 weak
  targets, 1 trial) that still draws proper strong-scaling **lines** (≥ 4 points/trace) and
  weak-scaling **curves** (≥ 3 traces/target). Use it to check the pipeline and that the
  graphs look right before committing to the overnight run. Its timings go to
  `timings_sample/` so they never mix with a full sweep. Set `SAMPLE_RUN = False` for the
  full matrix.
- **`DRY_RUN`** — a *true* dry run: §3 submits **nothing**, it just prints the jobs that
  *would* be launched. Works with either the sample or the full matrix.

The `plan.json` written here records the chosen weak targets so §5 draws exactly the curves
advertised below (no separate hard-coded target list).

`COLLECTION_MODE`: **STRONG** (each trace on the `NODE_COUNTS` grid within its bounds),
**WEAK** (each trace on a bespoke `nl = round(size / target)` so data/node ≈ constant — it
does *not* reuse the strong grid), or **ALL** (the union; collect once, draw everything).

**Why the weak targets take thought.** Trace sizes span ~0.7 GiB → 1.5 TiB, so no single
GiB/node target fits every trace (small traces would need `< 1` node; the 1.5 TiB trace
needs `≥ 64`). We therefore use several targets, each covering the subset of traces whose
integer node count is feasible — printed below. `NODE_BOUNDS` encodes memory *minimums* for
the largest traces and a *stepped ceiling*, clipped to `MAX_NODES = 256`.

**`WEAK_REQUIRED_TRACES`** — traces that MUST appear in the weak-scaling plan (e.g. the
292 GiB and 1.5 TiB traces), supplementing whatever `WEAK_TARGETS_GIB_PER_NODE` already
produces. For each required trace, extra targets are generated from *its own* natural node
counts (`NODE_BOUNDS ∩ NODE_COUNTS`), so it's guaranteed to be collected — at several node
counts if its grid has more than one — rather than left to chance. Other traces (including
the other required one) are matched in at each level using the normal `WEAK_TOLERANCE`; it's
expected and fine for a level to end up anchored on only one of the required traces if the
other doesn't fit that particular GiB/node ratio. Every required trace must also be listed
in `TRACES_TO_INCLUDE` (checked below) — nothing is added implicitly.

In [ ]:
# ============================ EXPERIMENT DESIGN ============================
SAMPLE_RUN = False   # True: small but real matrix to check the pipeline + graphs.
                     # False: the full sweep.
DRY_RUN    = True   # True: submit NOTHING in §3 — just print the jobs that WOULD run.
                     # (works with either the sample or the full matrix)

COLLECTION_MODE = "STRONG"                        # "STRONG" | "WEAK" | "ALL"
TRACES_TO_INCLUDE = [16, 32, 128, 384]

# Strong-scaling node grid (weak scaling computes its own node counts).
NODE_COUNTS = [1, 2, 4, 8, 16, 32, 64, 128, 256]

# Global node cap: the system has ~800 nodes, but we cap at 256 to keep runs affordable.
MAX_NODES = 256

# Per-trace (min_nodes, max_nodes):
#   min -> the two largest traces need enough nodes to fit in memory (cf. is_valid_combo)
#   max -> stepped "diminishing returns" ceiling that grows with trace size
NODE_BOUNDS = {
    2:   (1, 8),
    4:   (1, 16),
    8:   (1, 32),
    16:  (1, 32),
    32:  (1, 64),
    128: (16, 128),
    384: (64, 256),
}

# Weak-scaling targets (GiB/node): an explicit list, or "auto" to derive from the data.
WEAK_TARGETS_GIB_PER_NODE = "auto"
WEAK_TOLERANCE       = 0.05   # a trace joins curve g only if |actual - g| / g <= this
MIN_TRACES_PER_CURVE = 3     # a curve must cover >= this many traces -- ALWAYS enforced,
                              # including curves anchored on WEAK_REQUIRED_TRACES (3 is the
                              # hard floor; 4 is preferable and flagged in the printout).

# Traces that MUST ALL appear TOGETHER in every weak-scaling curve (e.g. the two largest),
# e.g. [128, 384]. Extra candidate targets are generated from each required trace's OWN
# achievable node counts (see required_trace_targets() below); a candidate only becomes a
# real curve if EVERY trace in this list matches it (within WEAK_TOLERANCE) AND the curve
# still has >= MIN_TRACES_PER_CURVE traces overall -- neither requirement is bypassed.
WEAK_REQUIRED_TRACES = [128, 384]

# Repetition & Slurm knobs
NUM_TRIALS       = 5               # trials PER config -- all run inside ONE Slurm job (§3),
                                   # so the queue is paid once per config, not once per trial
WARMUP           = False           # one unrecorded run per config before the timed trials
WALLTIME         = "00:30:00"      # PER-TRIAL budget; a config's job asks for NUM_TRIALS x this,
                                   # capped by Frontier's node-count walltime tiers (1-91n=2h,
                                   # 92-183n=6h, 184+n=12h) -- see job_walltime_for() in §3
EXCLUSIVE        = True            # #SBATCH --exclusive -> clean, non-shared perf numbers
KILL_ON_BAD_EXIT = True            # srun --kill-on-bad-exit=1

# ---- Sample-run overrides: a small but real matrix, sized so the graphs are actually
# useful -- each trace gets >= 4 strong-scaling node counts (real lines), and each weak
# target covers all 4 traces (real curves). Only the smallest/fastest traces are used.
# Everything else (mode, bounds, --exclusive, ...) is left as the full sweep would use it. ----
if SAMPLE_RUN:
    TRACES_TO_INCLUDE = [2, 4, 8, 16]          # small/fast traces
    NODE_COUNTS = [1, 2, 4, 8, 16]              # >= 4 strong points per trace -> real lines
    WEAK_TARGETS_GIB_PER_NODE = "auto"
    NUM_TRIALS = 1                              # one trial is enough for a look
    WALLTIME = "00:15:00"                       # small traces finish in well under this


def _bounds(traced_nodes):
    lo, hi = NODE_BOUNDS[traced_nodes]
    return lo, min(hi, MAX_NODES)


def weak_nodes(traced_nodes, g):
    """Bespoke node count for weak target g. Returns (nodes, actual_g, ok, reason)."""
    size = trace_sizes_gib[traced_nodes]
    lo, hi = _bounds(traced_nodes)
    n = max(1, int(round(size / g)))
    if n < lo:
        return n, size / n, False, f"n={n} < min {lo}"
    if n > hi:
        return n, size / n, False, f"n={n} > max {hi}"
    actual = size / n
    if abs(actual - g) / g > WEAK_TOLERANCE:
        return n, actual, False, f"actual {actual:.2f} >{WEAK_TOLERANCE:.0%} from {g}"
    return n, actual, True, "ok"


def auto_weak_targets():
    """Log-space scan of GiB/node; keep well-covered, geometrically-spaced targets."""
    lows = [trace_sizes_gib[t] / _bounds(t)[1] for t in TRACES_TO_INCLUDE]
    highs = [trace_sizes_gib[t] / _bounds(t)[0] for t in TRACES_TO_INCLUDE]
    picked = []
    for g in np.geomspace(min(lows), max(highs), 30):
        g = round(float(g), 2)
        cov = sum(weak_nodes(t, g)[2] for t in TRACES_TO_INCLUDE)
        if cov >= MIN_TRACES_PER_CURVE and (not picked or g / picked[-1] >= 1.7):
            picked.append(g)
    return picked


def required_trace_targets():
    """Candidate weak targets derived from each REQUIRED trace's own achievable node counts
    (NODE_BOUNDS ∩ NODE_COUNTS). These are only candidates -- a candidate only survives into
    weak_curves (below) if EVERY required trace actually matches it simultaneously."""
    targets = []
    for t in WEAK_REQUIRED_TRACES:
        lo, hi = _bounds(t)
        for n in NODE_COUNTS:
            if lo <= n <= hi:
                targets.append(round(trace_sizes_gib[t] / n, 4))
    return sorted(set(targets))


_missing_required = [t for t in WEAK_REQUIRED_TRACES if t not in TRACES_TO_INCLUDE]
if _missing_required:
    raise ValueError(f"WEAK_REQUIRED_TRACES {_missing_required} must also be listed in "
                      "TRACES_TO_INCLUDE.")

weak_targets = (auto_weak_targets() if WEAK_TARGETS_GIB_PER_NODE == "auto"
                else list(WEAK_TARGETS_GIB_PER_NODE))
weak_targets = sorted(set(weak_targets) | set(required_trace_targets()))


def weak_coverage_table(targets):
    rows = []
    for g in targets:
        for t in TRACES_TO_INCLUDE:
            n, actual, ok, reason = weak_nodes(t, g)
            rows.append({"target_GiB_per_node": g, "traced_nodes": t,
                         "size_GiB": round(trace_sizes_gib[t], 2), "nodes": n,
                         "actual_GiB_per_node": round(actual, 2),
                         "included": ok, "reason": reason})
    return pd.DataFrame(rows)


# ---- Build the (trace, nodes) configs for the chosen mode ----
strong_cfgs = set()
if COLLECTION_MODE in ("STRONG", "ALL"):
    for t in TRACES_TO_INCLUDE:
        lo, hi = _bounds(t)
        strong_cfgs |= {(t, n) for n in NODE_COUNTS if lo <= n <= hi}

# A weak-scaling candidate target only becomes a real curve if:
#  (1) EVERY trace in WEAK_REQUIRED_TRACES matches it (not just one of them), and
#  (2) the curve still has >= MIN_TRACES_PER_CURVE traces overall -- always enforced, never
#      bypassed just because a required trace is present.
# Near-duplicate candidates that resolve to the exact same (trace, nodes) members (common
# when several nearby g's all round to the same node counts) are deduped, keeping only one.
weak_cfgs, weak_curves = set(), {}
if COLLECTION_MODE in ("WEAK", "ALL"):
    _seen_member_sets = set()
    for g in weak_targets:
        members = []
        for t in TRACES_TO_INCLUDE:
            n, actual, ok, _ = weak_nodes(t, g)
            if ok:
                members.append((t, n))
        member_traces = {t for t, _ in members}
        has_all_required = all(t in member_traces for t in WEAK_REQUIRED_TRACES)
        meets_min = len(members) >= MIN_TRACES_PER_CURVE
        if not (has_all_required and meets_min):
            continue
        key = frozenset(members)
        if key in _seen_member_sets:
            continue   # identical curve already captured under a different (nearby) g
        _seen_member_sets.add(key)
        weak_curves[g] = members

    # Near-duplicate merge: the generic auto scan can rediscover almost the same level a
    # required-trace anchor already found (same traces, node counts off by ~1 from
    # rounding). Keep only the first (smallest-g) curve per distinct trace composition
    # within a small relative window -- two curves with the SAME traces but a genuinely
    # different target level (outside the window) are both kept.
    _dedup_radius = max(WEAK_TOLERANCE, 0.05)
    _kept_by_traceset = {}
    _deduped = {}
    for g in sorted(weak_curves):
        members = weak_curves[g]
        tset = frozenset(t for t, _ in members)
        prior_gs = _kept_by_traceset.get(tset, [])
        if any(abs(g - pg) / pg <= _dedup_radius for pg in prior_gs):
            continue
        _kept_by_traceset.setdefault(tset, []).append(g)
        _deduped[g] = members
    weak_curves = _deduped
    for members in weak_curves.values():
        weak_cfgs |= set(members)

configs = sorted(strong_cfgs | weak_cfgs)

# ---- One row per (trace, nodes) config. Trials are NOT expanded into separate rows any
# more: each config becomes ONE Slurm job that runs NUM_TRIALS trials internally (§3), so
# the queue is paid once per config instead of once per (config, trial). ----
runs_df = pd.DataFrame([
    {"traced_nodes": t, "nl": n,
     "trace_path": os.path.join(TRACE_INPUTS[t], TRACE_FILE),
     "size_GiB": round(trace_sizes_gib[t], 3),
     "gib_per_node": round(trace_sizes_gib[t] / n, 3),
     "in_strong": (t, n) in strong_cfgs, "in_weak": (t, n) in weak_cfgs}
    for (t, n) in configs
]).sort_values(["traced_nodes", "nl"]).reset_index(drop=True)

# ---- Persist the plan so §5 draws exactly these weak targets/curves (even for a previous
# run analysed in a fresh kernel), and so submission/analysis never drift apart. Also
# records the designed strong-scaling node grid so Graph 1's axis ticks stay anchored to
# it even when off-grid node counts (collected for weak-scaling on the same trace) are
# plotted alongside -- all real data is shown; only the tick labels are curated. ----
plan = {
    "collection_mode": COLLECTION_MODE,
    "traces": TRACES_TO_INCLUDE,
    "node_counts": NODE_COUNTS,
    "weak_targets": weak_targets,
    "weak_tolerance": WEAK_TOLERANCE,
    "min_traces_per_curve": MIN_TRACES_PER_CURVE,
    "weak_required_traces": WEAK_REQUIRED_TRACES,
    "weak_curves": {str(g): members for g, members in weak_curves.items()},
    "num_trials": NUM_TRIALS,
    "sample_run": SAMPLE_RUN,
    "dry_run": DRY_RUN,
}
(RUN_DIR / "plan.json").write_text(json.dumps(plan, indent=2))

# ---- Report the plan (so you see the cost before submitting) ----
n_configs = len(configs)
n_jobs = n_configs + (n_configs if WARMUP else 0)   # one Slurm job per config (+ warmup)
print("*** SAMPLE RUN (small, real) ***" if SAMPLE_RUN else "*** FULL SWEEP ***",
      "   [DRY RUN — nothing will be submitted]" if DRY_RUN else "")
print(f"MODE = {COLLECTION_MODE}   traces = {TRACES_TO_INCLUDE}")
print(f"unique (trace, nodes) configs = {n_configs}   trials/config = {NUM_TRIALS} "
      f"(run back-to-back IN one job)   total Slurm jobs = {n_jobs}"
      f"{' (incl. warmup)' if WARMUP else ''}")
print(f"weak target candidates (GiB/node) = {weak_targets}")
if WEAK_REQUIRED_TRACES:
    print(f"weak-required traces = {WEAK_REQUIRED_TRACES} "
          "(every weak curve below must contain ALL of these)")
if COLLECTION_MODE in ("WEAK", "ALL"):
    if not weak_curves:
        print("\n*** No weak-scaling curves survived (need ALL required traces + "
              f">= {MIN_TRACES_PER_CURVE} traces total). Try loosening WEAK_TOLERANCE "
              "or check NODE_BOUNDS overlap between the required traces. ***")
    else:
        print("weak curves that will be plotted:")
        for g, members in sorted(weak_curves.items()):
            pts = ", ".join(f"s{t}->{n}n(~{trace_sizes_gib[t] / n:.2f})"
                            for t, n in sorted(members))
            n_members = len(members)
            pref = "" if n_members >= max(MIN_TRACES_PER_CURVE, 4) else " (min met, not 4+)"
            print(f"  {g:>7.3f} GiB/node [{n_members} traces]{pref} : {pts}")
print("\nWeak-scaling coverage (audit — why each trace is in/out per target):")
display(weak_coverage_table(weak_targets))
print("First planned runs:")
display(runs_df.head(12))


# 3. Build & submit the sweep

Each config becomes one Slurm job: `e4s-cl launch … srun … <converter> -nl<N> <trace>
--timings-csv <dir>`. Jobs are **dependency-chained** (`--dependency afterany:<prev>`) so
they run one at a time — clean, non-overlapping timings with no polling loop.

- **`DRY_RUN = True`** → this cell **submits nothing**; it just prints each job that would be
  launched, including the exact `sbatch` command and generated script contents (so the
  underlying `e4s-cl launch … srun …` command can be copy-pasted straight into a debug QOS
  job to sanity-check it). Nothing is written, no `sbatch` is called.
- **`SAMPLE_RUN`** → jobs/folders are tagged `sample-…`, Parquet output goes under `pq/`, and
  timings go to `timings_sample/` (kept separate from a full sweep's `timings/`).


Per-run artifacts (the `sbatch` script, `run_args.json`, `converter.log`) are written under
`run_logs/`, and a `manifest*.csv` records every job id.

**Sanity-checking a new `SYSTEM`/container in the debug QOS.** Run this cell with
`DRY_RUN = True`: alongside the per-job preview, it prints a ready-to-paste `salloc` command
plus the single `e4s-cl launch … srun …` line for the smallest planned trace, with
`THREAD_COUNT`, `SLURM_ACCOUNT`, the container path, and the trace path all filled in from
this session -- nothing to hand-edit. Just:

1. Copy the printed `salloc …` line into your shell and wait for the allocation.
2. Once inside it, paste the printed `e4s-cl launch …` line.

If you'd rather test the *exact* `sbatch` artifact instead, save the per-job `script
contents:` block to a file and `sbatch` it using that job's printed `sbatch cmd:` flags plus
`--qos=debug` (neither the dry-run output nor `SBATCH_EXTRA_ARGS` include a QOS by default).

Once `e4s-cl launch` succeeds for that small trace, there's no need to separately validate the
`sbatch` wrapper -- set `DRY_RUN = False` and let this cell submit the real sweep.

In [ ]:
def _hms_to_s(hms):
    h, m, s = (int(x) for x in hms.split(":"))
    return h * 3600 + m * 60 + s


def _s_to_hms(sec):
    sec = int(sec)
    return f"{sec // 3600:02d}:{(sec % 3600) // 60:02d}:{sec % 60:02d}"


def _tier_cap_h(nl):
    """Frontier per-job max walltime by node-count bin (batch scheduling policy):
    1-91 nodes -> 2h, 92-183 -> 6h, 184+ -> 12h."""
    return 2 if nl <= 91 else (6 if nl <= 183 else 12)


def job_walltime_for(nl, n_trials):
    """(HH:MM:SS, capped?) for a job running n_trials back-to-back. Each trial is
    budgeted WALLTIME; the job needs n_trials x that, but Frontier caps it by node
    count -- if the budget exceeds the cap, later trials may be killed (analysis just
    sees fewer trials for that config)."""
    desired = n_trials * _hms_to_s(WALLTIME)
    cap = _tier_cap_h(nl) * 3600
    return _s_to_hms(min(desired, cap)), desired > cap


def build_launch_cmd(nl, trace_path, converter_log, output_dir, timings_subdir):
    """The single e4s-cl launch + srun + converter command for ONE trial (per-trial
    srun --time = WALLTIME). This is also what the debug-QOS copy-paste line uses."""
    srun_args = [f"--nodes={nl}", "--ntasks-per-node=1", f"--cpus-per-task={THREAD_COUNT}",
                 f"--time={WALLTIME}", f"--output={converter_log}"]
    if EXCLUSIVE:
        srun_args.append("--exclusive")
    if KILL_ON_BAD_EXIT:
        srun_args.append("--kill-on-bad-exit=1")
    converter_args = [f"-nl{nl}", trace_path, f"--strategy={STRATEGY}", f"--format={FORMAT}",
                      f"--outputDir={output_dir}", "--log=WARN", f"--timings-csv {timings_subdir}"]
    return " ".join(["e4s-cl", "launch", f"--image={container_path}", "srun", *srun_args,
                     "--", entrypoint_path, *converter_args])


def build_job_script(traced_nodes, nl, trace_path, per_run, output_dir, timings_root,
                     n_trials, jobtime_file=None):
    """Per-config sbatch script: run `n_trials` converter trials sequentially in ONE
    allocation (one queue wait for all trials, not one per trial). Each trial writes its
    timings to timings_root/size{t}_nl{n}_trial{k}, so the §5 loader still sees separate
    trials; the throwaway Parquet outputDir is cleared between trials to bound disk use.
    When jobtime_file is given, the job records its overall wall window for the §5 overlap
    check (written OUTSIDE the timings/ tree)."""
    lines = ["#!/bin/bash", "set -u"]
    if jobtime_file:
        lines += [
            f'_jt="{jobtime_file}"',
            'mkdir -p "$(dirname "$_jt")"',
            'echo "jobid=${SLURM_JOB_ID} host=$(hostname)'
            f' nodes={nl} trials={n_trials} start=$(date +%s.%N) start_iso=$(date -Is)" > "$_jt"',
        ]
    lines.append("_rc=0")
    for k in range(1, n_trials + 1):
        tag = f"size{traced_nodes}_nl{nl}_trial{k}"
        clog = str(Path(per_run) / f"converter_trial{k}.log")
        launch = build_launch_cmd(nl, trace_path, clog, str(output_dir),
                                   f"{timings_root}/{tag}")
        lines += [
            f'echo "=== trial {k}/{n_trials} : {tag} ==="',
            f'rm -rf "{output_dir}"; mkdir -p "{output_dir}"',   # bound disk: reuse 1 Parquet dir
            launch,
            'rc=$?; if [ "$rc" -ne 0 ]; then _rc="$rc"; fi',
        ]
    if jobtime_file:
        lines.append('echo "end=$(date +%s.%N) end_iso=$(date -Is) rc=${_rc}" >> "$_jt"')
    lines.append('exit "$_rc"')
    return "\n".join(lines) + "\n"


def submit_one(traced_nodes, nl, prev_job_id, warmup=False):
    # One job per (trace, nodes) config; it runs NUM_TRIALS trials internally. Trial dirs
    # stay regex-parseable (size{N}_nl{M}_trial{K}) for the analysis loader.
    core = f"size{traced_nodes}_nl{nl}"
    label = ("sample-" if SAMPLE_RUN else "") + core
    trace_path = os.path.join(TRACE_INPUTS[traced_nodes], TRACE_FILE)
    job_name = f"f2-{'sample-' if SAMPLE_RUN else ''}s{traced_nodes}-nl{nl}"
    n_trials = 1 if warmup else NUM_TRIALS

    # warmup timings go to a throwaway dir (never analysed); real trials go to the sweep root.
    timings_root = (RUN_DIR / "warmup_timings") if warmup else (
        SAMPLE_TIMINGS_DIR if SAMPLE_RUN else TIMINGS_DIR)
    output_dir = OUTPUT_ROOT / label
    per_run = RUN_LOG_DIR / label
    script_path = per_run / "sbatch_script.sh"

    # Self-reported run-window file for the §5 overlap check. Kept in a sibling job_times/
    # folder (NOT under timings/), so load_timings never picks it up. Warmups are untracked.
    jobtime_file = None if warmup else (RUN_DIR / "job_times" / f"{core}.txt")

    job_script = build_job_script(traced_nodes, nl, trace_path, str(per_run),
                                  str(output_dir), str(timings_root), n_trials,
                                  jobtime_file=str(jobtime_file) if jobtime_file else None)
    job_wt, capped = job_walltime_for(nl, n_trials)
    sbatch_cmd = ["sbatch", "--parsable", f"--job-name={job_name}",
                  f"--nodes={nl}", "--ntasks-per-node=1", f"--cpus-per-task={THREAD_COUNT}",
                  f"--time={job_wt}", f"--output={SLURM_LOG_DIR / (label + '-%j.out')}"]
    if EXCLUSIVE:
        sbatch_cmd.append("--exclusive")
    sbatch_cmd += SBATCH_EXTRA_ARGS
    if prev_job_id is not None:
        sbatch_cmd += ["--dependency", f"afterany:{prev_job_id}"]

    # True dry run: report the job and do nothing else (no dirs, no files, no sbatch).
    if DRY_RUN:
        print(f"  [dry] would submit {job_name}: -nl{nl}, "
              f"trace s{traced_nodes} ({trace_sizes_gib[traced_nodes]:.2f} GiB), "
              f"{n_trials} trial(s) in one job, --time {job_wt} (tier cap {_tier_cap_h(nl)}h)")
        if capped:
            print(f"        *** WARNING: {n_trials}x{WALLTIME} exceeds the {_tier_cap_h(nl)}h "
                  f"cap for {nl} nodes -- late trials may be killed. Reduce NUM_TRIALS or "
                  f"WALLTIME for small-node configs if you need all trials. ***")
        print(f"        sbatch cmd: {' '.join(sbatch_cmd)} {script_path}")
        print(f"        script contents:\n" + "\n".join(
            f"          {line}" for line in job_script.splitlines()))
        return "DRYRUN"

    per_run.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    script_path.write_text(job_script)
    (per_run / "run_args.json").write_text(json.dumps(
        {"traced_nodes": traced_nodes, "nl": nl, "num_trials": n_trials, "warmup": warmup,
         "sample_run": SAMPLE_RUN, "trace_path": trace_path,
         "strategy": STRATEGY, "format": FORMAT}, indent=2))

    sbatch_cmd.append(str(script_path))
    (per_run / "submission_command.txt").write_text(" ".join(sbatch_cmd) + "\n")

    res = subprocess.run(sbatch_cmd, capture_output=True, text=True)
    if res.returncode != 0:
        workflows.print_and_log(f"[{label}] sbatch FAILED: {res.stderr.strip()}")
        return None
    return res.stdout.strip().split(";")[0]


# ---- Submit everything, dependency-chained (or preview, if DRY_RUN) ----
if DRY_RUN:
    print(f"*** DRY RUN — nothing will be submitted *** {len(runs_df)} jobs "
          f"({NUM_TRIALS} trial(s) each) would launch:")
else:
    print(f"*** {'SAMPLE RUN' if SAMPLE_RUN else 'FULL SWEEP'} *** submitting "
          f"{len(runs_df) + (len(configs) if WARMUP else 0)} jobs "
          f"({NUM_TRIALS} trial(s) each, dependency-chained)")

# ---- Debug-QOS sanity check: a fully filled-in, copy-pasteable salloc + launch command for
# the smallest planned trace, so there's nothing to hand-edit before pasting it in. ----
if DRY_RUN and not runs_df.empty:
    _dbg_row = runs_df.iloc[0]
    _dbg_t, _dbg_n = int(_dbg_row.traced_nodes), int(_dbg_row.nl)
    _dbg_time = "00:30:00"
    _dbg_dir = RUN_DIR / "debug_check"
    _dbg_dir.mkdir(parents=True, exist_ok=True)
    _dbg_launch = build_launch_cmd(_dbg_n, _dbg_row.trace_path, str(_dbg_dir / "converter.log"),
                                   str(_dbg_dir / "pq"), str(_dbg_dir / "timings"))
    _salloc_parts = ["salloc", f"--nodes={_dbg_n}", "--ntasks-per-node=1",
                     f"--cpus-per-task={THREAD_COUNT}", f"--time={_dbg_time}", "--qos=debug"]
    if SLURM_ACCOUNT:
        _salloc_parts.append(f"--account={SLURM_ACCOUNT}")
    if EXCLUSIVE:
        _salloc_parts.append("--exclusive")
    print(f"\n*** Debug-QOS sanity check (copy-paste; smallest planned trace: "
          f"s{_dbg_t} @ nl{_dbg_n}) ***")
    print("  " + " ".join(_salloc_parts))
    print("  # once inside the allocation, run:")
    print("  " + _dbg_launch)

prev_job_id = None
manifest_rows = []

if WARMUP and not DRY_RUN:
    for (t, n) in configs:
        prev_job_id = submit_one(t, n, prev_job_id, warmup=True) or prev_job_id

for _, r in runs_df.iterrows():
    jid = submit_one(int(r.traced_nodes), int(r.nl), prev_job_id)
    if jid and jid != "DRYRUN":
        prev_job_id = jid
    manifest_rows.append({**r.to_dict(), "num_trials": NUM_TRIALS,
                          "sample_run": SAMPLE_RUN, "job_id": jid})

manifest = pd.DataFrame(manifest_rows)
manifest_name = ("manifest_dryrun.csv" if DRY_RUN
                 else "manifest_sample.csv" if SAMPLE_RUN else "manifest.csv")
manifest.to_csv(RUN_DIR / manifest_name, index=False)
if DRY_RUN:
    print(f"\nDry run complete — no jobs submitted. Preview manifest: {RUN_DIR / manifest_name}")
else:
    n_ok = (manifest["job_id"].notna() & (manifest["job_id"] != "DRYRUN")).sum()
    print(f"Submitted {n_ok} / {len(manifest)} jobs "
          f"({NUM_TRIALS} trial(s) each, dependency-chained). "
          f"Manifest: {RUN_DIR / manifest_name}")
display(manifest.head(12))


# 4. Monitor the queue (optional)

Jobs run serially via the dependency chain, so timing CSVs appear one-by-one under
`timings/`. Use the widget below to watch the queue, or run
`workflows.wait_until_my_jobs_finished()` to block until the sweep is done. Nothing here is
required for the analysis — it just reads whatever CSVs already exist.

In [ ]:
from workflows import watch_queue_widget, wait_until_my_jobs_finished

# Live squeue monitor (press Start). Safe to skip.
watch_queue_widget()

# Or, to block until everything finishes, run:
# wait_until_my_jobs_finished()

# 5. Results analysis

**Self-contained** — this half only needs a run's timings folder, so you can restart the
kernel and run just §5 onward.

- To analyse the run you just submitted, leave **`ANALYZE_RUN = None`** (the default).
- To analyse a **previous** run *without collecting new data*, set **`ANALYZE_RUN`** to that
  run's tag (e.g. `"run_20260714_101500"`) or a full path to its folder.
- **`ANALYZE_SAMPLE`** picks the sample timings (`timings_sample/`) vs the full sweep's
  (`timings/`); it defaults to whatever `SAMPLE_RUN` was this session.

Weak-scaling targets are read from the run's `plan.json`, so the graphs draw **exactly** the
curves §2 advertised (no separate hard-coded list). Trace sizes come from the run's cached
`trace_sizes.json`; timings come straight from the converter's CSVs (`run_*.csv`,
`tasks_*.csv`, `phases_*.csv`) — no log parsing.

In [ ]:
# --- Analysis configuration (self-contained: this half runs on its own) ---
import re, json
from pathlib import Path
import numpy as np
import pandas as pd
from plotnine import *  # noqa: F401,F403

# === Which run to analyse? =================================================
# None  -> the run created above in this kernel session (needs §0-§3 to have run).
# tag   -> a previous run's tag, e.g. "run_20260714_101500"
# path  -> a full path to a run folder OR directly to its timings/ folder
# Use this to analyse an existing run WITHOUT collecting new data: restart the
# kernel, set ANALYZE_RUN here, then run this cell and the graph cells below.
ANALYZE_RUN = None
ANALYZE_RUN = "/lus/bnchlu1/khandeka/dev/hpc-ai-adv-dev/fastotf2/workflows/scaling/out/run_20260714_043638_save"

# Read the small SAMPLE run's timings (timings_sample/) or the full sweep's (timings/)?
# Defaults to whatever SAMPLE_RUN was in this session, else the full-sweep timings.
ANALYZE_SAMPLE = globals().get("SAMPLE_RUN", False)

# Where run folders live — used to turn a bare ANALYZE_RUN tag into a path.
OUT_ROOT = (WORKFLOW_DIR / "out") if "WORKFLOW_DIR" in globals() else Path(
    "/lus/bnchlu1/khandeka/dev/hpc-ai-adv-dev/fastotf2/workflows/scaling/out")


def resolve_timings_dir(analyze_run, sample):
    """Resolve the timings folder to analyse from a tag, a path, or the live session."""
    sub = "timings_sample" if sample else "timings"
    if analyze_run is None:                                  # this session's run
        if "TIMINGS_DIR" in globals():
            return SAMPLE_TIMINGS_DIR if sample else TIMINGS_DIR
        raise RuntimeError("No run in this kernel — set ANALYZE_RUN to a past run tag/path.")
    p = Path(analyze_run)
    if p.exists():                                            # a full path was given
        return p if p.name.startswith("timings") else (p / sub)
    return OUT_ROOT / str(analyze_run) / sub                 # a bare run tag


HOST_TIMINGS_DIR = resolve_timings_dir(ANALYZE_RUN, ANALYZE_SAMPLE)
RUN_DIR_ANALYSIS = Path(HOST_TIMINGS_DIR).parent
if not Path(HOST_TIMINGS_DIR).exists():
    raise FileNotFoundError(f"Timings folder not found: {HOST_TIMINGS_DIR}\n"
                            "Check ANALYZE_RUN / ANALYZE_SAMPLE.")

# Trace sizes (GiB) for GiB/node derivations — from the run's cached JSON.
_sizes_json = RUN_DIR_ANALYSIS / "trace_sizes.json"
if _sizes_json.exists():
    trace_sizes_gib = {int(k): v for k, v in json.loads(_sizes_json.read_text()).items()}
elif "trace_sizes_gib" not in globals():
    raise RuntimeError(f"No trace_sizes.json at {_sizes_json}; set trace_sizes_gib manually.")

# plan.json from the run that produced this data (used below just for the strong-scaling
# node grid; Graph 3's weak-scaling analysis derives its own targets directly from the
# collected data instead of relying on the original design-time targets).
_plan = {}
_plan_json = RUN_DIR_ANALYSIS / "plan.json"
if _plan_json.exists():
    _plan = json.loads(_plan_json.read_text())

# The strong-scaling node grid this run was designed around (from plan.json). Graph 1 uses
# this only to choose which x-axis ticks to *label* -- every real measurement is still
# plotted (including any off-grid node counts collected for weak-scaling on the same
# trace); they just land at their correct position without cluttering the axis with a tick.
NODE_COUNTS_ANALYSIS = _plan.get("node_counts") or globals().get("NODE_COUNTS")

# Optional figure export (sample plots kept separate so they never overwrite real figures)
SAVE_PLOTS = True
PLOTS_DIR = RUN_DIR_ANALYSIS / ("plots_sample" if ANALYZE_SAMPLE else "plots")


def _maybe_save(p, name):
    if SAVE_PLOTS:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        p.save(PLOTS_DIR / f"{name}.png", dpi=150, verbose=False)


# ---- Shared plot helpers (labels, axes, units) ----------------------------
SEC_PER_MIN = 60.0


def size_label(traced_nodes):
    """Human label for a trace: its on-disk size in GiB (or TiB for the biggest)."""
    g = trace_sizes_gib[traced_nodes]
    if g >= 1024:
        return f"{g / 1024:.2f} TiB"
    if g >= 10:
        return f"{g:.0f} GiB"
    return f"{g:.1f} GiB"


def trace_cat(traced_series):
    """Ordered categorical of size labels (ascending by real size) for stable legends."""
    present = set(traced_series)
    order = [size_label(t) for t in sorted(trace_sizes_gib, key=lambda k: trace_sizes_gib[k])
             if t in present]
    return pd.Categorical(traced_series.map(size_label), categories=order, ordered=True)


def node_axis(nodes):
    """log2-spaced x-axis but with plain integer tick labels (no 2^ ticks). Use for
    STRONG scaling, whose NODE_COUNTS form a real power-of-two grid."""
    ns = sorted({int(n) for n in nodes})
    return scale_x_continuous(trans="log2", breaks=ns, labels=[str(n) for n in ns])


def node_axis_linear(nodes):
    """Linear x-axis with integer breaks at exactly the node counts present. Use for WEAK
    scaling: its node counts are bespoke (round(size/target) per trace), not a doubling
    sequence, so a log2 axis would space them unevenly and look distorted."""
    ns = sorted({int(n) for n in nodes})
    return scale_x_continuous(breaks=ns, labels=[str(n) for n in ns])


print(f"Analysing: {HOST_TIMINGS_DIR}"
      f"{'  [sample]' if ANALYZE_SAMPLE else ''}"
      f"{'  (previous run)' if ANALYZE_RUN is not None else '  (this session)'}")

In [ ]:
def _parse_tag(name):
    m = re.match(r"size(\d+)_nl(\d+)_trial(\d+)", name)
    return dict(traced_nodes=int(m[1]), nl=int(m[2]), trial=int(m[3])) if m else {}


def _ts(name):
    m = re.search(r"(?:run|tasks|phases)_(.+)\.csv$", name)
    return m.group(1) if m else None


def load_timings(base_dir):
    """Read run_/tasks_/phases_ CSVs under base_dir; add traced_nodes/nl/trial + sizes."""
    base = Path(base_dir)
    out = {"run": [], "tasks": [], "phases": []}
    for kind in out:
        for csv in sorted(base.rglob(f"{kind}_*.csv")):
            meta = _parse_tag(csv.parent.name)
            if not meta:
                continue                       # skip warmup_/other dirs
            df = pd.read_csv(csv)
            for k, v in meta.items():
                df[k] = v
            df["timestamp"] = _ts(csv.name)
            out[kind].append(df)
    frames = {k: (pd.concat(v, ignore_index=True) if v else pd.DataFrame())
              for k, v in out.items()}
    for df in frames.values():
        if not df.empty and "traced_nodes" in df:
            df["data_size_gib"] = df["traced_nodes"].map(trace_sizes_gib)
            df["gib_per_node"] = df["data_size_gib"] / df["nl"]
    return frames["run"], frames["tasks"], frames["phases"]


df_runs, df_tasks, df_phases = load_timings(HOST_TIMINGS_DIR)
print(f"runs={len(df_runs)}  tasks={len(df_tasks)}  phases={len(df_phases)}")
if not df_runs.empty:
    print("traces:", sorted(df_runs.traced_nodes.unique()),
          "  node counts:", sorted(df_runs.nl.unique()))
    display(df_runs.head())
else:
    print("No timing CSVs found yet — jobs may still be running.")

## Data-validity check — did any two jobs overlap?

Before trusting the graphs, confirm the sweep ran **serially** — these runs are I/O-heavy,
so two overlapping jobs would contend for the shared filesystem and corrupt each other's
timings. Two **independent** sources are cross-checked for the run being analysed
(`RUN_DIR_ANALYSIS`): each job's *self-reported* wall window (`job_times/`, written by its
own sbatch script) and **`sacct`**. Any overlapping pair is printed so you can discard and
re-run just those configs.

Like the rest of §5 this is **self-contained** — restart the kernel, set `ANALYZE_RUN`, run
the §5 config cell, then this one. It reads only `job_times/` + `manifest` + `sacct`; it
never touches the timing DataFrames, so it can't change any graph. (Older runs predating the
`job_times/` feature simply report "no run windows available".)


In [ ]:
# ---- Data-validity check: did any two of this run's jobs run at the SAME time? ----
# These runs are I/O-heavy, so two overlapping jobs would contend for the shared filesystem
# and corrupt each other's timings. This validates the run was serial, using TWO independent
# sources: (a) each job's self-reported wall window (RUN_DIR_ANALYSIS/job_times/, written by
# its own sbatch script) and (b) sacct. It reads ONLY job_times/ + manifest + sacct for the
# run being analysed (RUN_DIR_ANALYSIS) -- it does not touch df_runs/df_tasks/df_phases, so
# it cannot change any graph. Self-contained: run it after the §5 config cell on any run.
import subprocess as _sp


def _overlap_ids(run_dir):
    cands = [c for c in sorted(Path(run_dir).glob("manifest*.csv")) if "dryrun" not in c.name]
    if not cands:
        return []
    m = pd.read_csv(cands[0])
    return [str(j).split(";")[0] for j in m.get("job_id", [])
            if str(j) not in ("", "nan", "None", "DRYRUN")]


def _iv_jobtimes(run_dir):
    """Intervals from the jobs' own start/end reports (run_dir/job_times/*.txt)."""
    rows = []
    jt = Path(run_dir) / "job_times"
    if jt.is_dir():
        for f in sorted(jt.glob("*.txt")):
            g = dict(re.findall(r"(\w+)=(\S+)", f.read_text()))
            if "start" in g and "end" in g:
                rows.append({"tag": f.stem, "jobid": g.get("jobid", ""),
                             "start": float(g["start"]), "end": float(g["end"])})
    return pd.DataFrame(rows)


def _iv_sacct(ids):
    """Intervals from sacct for the manifest's job ids (empty if sacct/ids unavailable)."""
    if not ids:
        return pd.DataFrame()
    res = _sp.run(["sacct", "-X", "-j", ",".join(ids), "--parsable2", "--noheader",
                   "--format=JobIDRaw,JobName,Start,End,State"],
                  capture_output=True, text=True)
    rows = []
    for line in res.stdout.strip().splitlines():
        p = line.split("|")
        if len(p) < 5:
            continue
        jid, name, start, end, state = p[:5]
        s, e = pd.to_datetime(start, errors="coerce"), pd.to_datetime(end, errors="coerce")
        if pd.notna(s) and pd.notna(e):
            rows.append({"tag": name, "jobid": jid, "start": s.timestamp(),
                         "end": e.timestamp(), "state": state})
    return pd.DataFrame(rows)


def find_overlaps(df):
    """Return (overlapping-pairs DataFrame, max_concurrency). O(n^2) -- fine for a sweep."""
    if df.empty:
        return pd.DataFrame(columns=["a", "b", "overlap_s"]), 0
    df = df.sort_values("start").reset_index(drop=True)
    pairs = []
    for i in range(len(df)):
        for j in range(i + 1, len(df)):
            lo = max(df.at[i, "start"], df.at[j, "start"])
            hi = min(df.at[i, "end"], df.at[j, "end"])
            if hi > lo:
                pairs.append({"a": df.at[i, "tag"], "b": df.at[j, "tag"],
                              "overlap_s": round(hi - lo, 1)})
    events = sorted([(r.start, 1) for r in df.itertuples()] +
                    [(r.end, -1) for r in df.itertuples()])
    cur = mx = 0
    for _, delta in events:
        cur += delta
        mx = max(mx, cur)
    return pd.DataFrame(pairs), mx


def check_overlaps(run_dir=None):
    """Validate the analysed run was serial. Defaults to RUN_DIR_ANALYSIS (the §5 run)."""
    run_dir = run_dir or RUN_DIR_ANALYSIS
    ids = _overlap_ids(run_dir)
    for label, df in (("job-script self-report", _iv_jobtimes(run_dir)),
                      ("sacct", _iv_sacct(ids))):
        pairs, mx = find_overlaps(df)
        if df.empty:
            print(f"[{label}] no run windows available (older run, or jobs not finished).")
            continue
        span_h = (df["end"].max() - df["start"].min()) / 3600.0
        print(f"[{label}] {len(df)} jobs, wall span {span_h:.2f} h, max concurrency = {mx}")
        if pairs.empty:
            print("   ✓ no overlaps — every run was serial (clean timings).")
        else:
            print(f"   ✗ {len(pairs)} overlapping pair(s) — DISCARD & re-run these configs:")
            display(pairs.sort_values("overlap_s", ascending=False))


check_overlaps()


## Graph 1 — Strong Scaling

Fixed trace, more nodes. **Every real measurement is plotted** — including node counts that
were originally chosen to hit a weak-scaling target (e.g. `nl=11` for an 8-node trace): it's
still the same fixed trace measured with a different node count, exactly what strong scaling
means, so there's no reason to drop it. The x-axis is log2 (node counts double at each grid
step), with tick **labels** anchored to the originally-designed grid (`plan.json`'s
`node_counts`) so off-grid points don't clutter it with an extra tick.

**`STRONG_EXCLUDE_TRACES`** drops the smallest trace(s) from these three plots only (default:
the two smallest) — they don't have enough work to show any benefit from extra nodes, and
including them distorts the rest of the curves.

Traces are labelled by their on-disk size; time is in minutes. **Total time and throughput**
use a **linear** y-axis (no log-log, per request). **Speedup** stays **log-log** — since
ideal speedup is linear in node count, a log-log plot turns it into a straight diagonal,
making it easy to compare against the dashed ideal line.

In [ ]:
# Exclude traces too small to benefit meaningfully from more nodes -- keeping them in
# distorts the rest of the strong-scaling curves. Applies to Graph 1 only.
STRONG_EXCLUDE_TRACES = {2, 4, 8}   # ~0.7 GiB and ~2.0 GiB traces


def _agg(df, value):
    g = (df.groupby(["traced_nodes", "nl"])[value]
           .agg(med="median", lo="min", hi="max").reset_index())
    g["trace"] = trace_cat(g["traced_nodes"])
    return g


def minutes_axis(vmax):
    """Denser y-axis breaks for a minutes scale: 1-minute steps by default, only
    backing off to a coarser step (2, 5, 10, ...) if 1-minute ticks would be too
    dense to read."""
    for step in (1, 2, 5, 10, 15, 30, 60):
        if vmax / step <= 20:
            return scale_y_continuous(breaks=np.arange(0, vmax + step, step))
    return scale_y_continuous(breaks=np.arange(0, vmax + 60, 60))


def speedup_axis(vmax):
    """Denser log2 y-axis with plain-number labels (not '2^x' notation) -- two
    ticks per octave (e.g. 4, 6, 8, 12, 16) instead of only powers of two."""
    candidates = [1, 1.5, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256,
                  384, 512, 768, 1024]
    breaks = [b for b in candidates if b <= vmax * 1.15] or candidates[:2]
    return scale_y_continuous(trans="log2", breaks=breaks,
                              labels=[f"{b:g}" for b in breaks])


# Tick marks for the node axis: the originally-designed strong-scaling grid, so off-grid
# node counts collected for weak-scaling on the same trace don't clutter the axis with
# their own tick label -- they're still plotted, just without one.
_node_ticks = NODE_COUNTS_ANALYSIS or (sorted(df_runs["nl"].unique()) if not df_runs.empty else [])

if not df_runs.empty:
    d = df_runs[~df_runs["traced_nodes"].isin(STRONG_EXCLUDE_TRACES)].copy()
    if d.empty:
        print("Nothing left to plot after STRONG_EXCLUDE_TRACES filtering.")
    else:
        d["time_min"] = d["totalTime"] / SEC_PER_MIN
        d["thr_Mevents"] = d["throughput"] / 1e6

        # G1a — total time (minutes), linear y-axis with dense (>= 1-2 min) breaks
        s = _agg(d, "time_min")
        p1 = (ggplot(s, aes("nl", "med", color="trace", fill="trace"))
              + geom_ribbon(aes(ymin="lo", ymax="hi"), alpha=0.15, color=None)
              + geom_line() + geom_point(size=2)
              + node_axis(_node_ticks) + expand_limits(y=0)
              + minutes_axis(float(s["hi"].max()))
              + scale_color_brewer(type="qual", palette="Dark2")
              + scale_fill_brewer(type="qual", palette="Dark2")
              + labs(title="Strong Scaling: Conversion Time", x="Number of Nodes",
                     y="Total Time (minutes)", color="Trace Size", fill="Trace Size")
              + theme_bw() + theme(figure_size=(9, 5)))
        display(p1); _maybe_save(p1, "g1a_strong_time")

        # G1b — speedup vs each trace's smallest node count, log-log (dashed = ideal diagonal)
        parts = []
        for _, sub in s.groupby("trace", observed=True):
            sub = sub.sort_values("nl").copy()
            sub["speedup"] = sub["med"].iloc[0] / sub["med"]
            sub["ideal"] = sub["nl"] / sub["nl"].iloc[0]
            parts.append(sub)
        sp = pd.concat(parts)
        p2 = (ggplot(sp, aes("nl", "speedup", color="trace"))
              + geom_line(aes(y="ideal", group="trace"), color="grey", linetype="dashed")
              + geom_line() + geom_point(size=2)
              + node_axis(_node_ticks)
              + speedup_axis(float(max(sp["speedup"].max(), sp["ideal"].max())))
              + scale_color_brewer(type="qual", palette="Dark2")
              + labs(title="Strong Scaling: Speedup (log-log; dashed = ideal)",
                     x="Number of Nodes", y="Speedup vs. Smallest Node Count",
                     color="Trace Size")
              + theme_bw() + theme(figure_size=(9, 5)))
        display(p2); _maybe_save(p2, "g1b_strong_speedup")

        # G1c — throughput (million events / s); linear, y-from-0
        st = _agg(d, "thr_Mevents")
        p3 = (ggplot(st, aes("nl", "med", color="trace", fill="trace"))
              + geom_ribbon(aes(ymin="lo", ymax="hi"), alpha=0.15, color=None)
              + geom_line() + geom_point(size=2)
              + node_axis(_node_ticks) + expand_limits(y=0)
              + scale_color_brewer(type="qual", palette="Dark2")
              + scale_fill_brewer(type="qual", palette="Dark2")
              + labs(title="Strong Scaling: Throughput", x="Number of Nodes",
                     y="Throughput (Million Events / s)", color="Trace Size", fill="Trace Size")
              + theme_bw() + theme(figure_size=(9, 5)))
        display(p3); _maybe_save(p3, "g1c_strong_throughput")
else:
    print("No runs to plot.")

## Graph 2 — Multi-Trial Consistency

Run-to-run spread across trials, faceted by trace size so the panel stays a manageable size
no matter how many node counts or trials were collected (a single combined axis previously
grew unbounded and could exceed plotnine's figure-size limit). A tight box means the timing
is reproducible.

In [ ]:
if not df_runs.empty:
    d = df_runs.copy()
    d["n_trials"] = d.groupby(["traced_nodes", "nl"])["trial"].transform("nunique")
    d = d[d["n_trials"] > 1].copy()
    if not d.empty:
        d["time_min"] = d["totalTime"] / SEC_PER_MIN
        d["trace"] = trace_cat(d["traced_nodes"])
        n_facets = d["trace"].nunique()
        ncol = min(4, n_facets) if n_facets else 1
        nrow = max(1, -(-n_facets // ncol)) if ncol else 1
        p = (ggplot(d, aes("factor(nl)", "time_min"))
             + geom_boxplot(outlier_alpha=0.0)
             + geom_jitter(width=0.15, height=0, alpha=0.5, size=1.3)
             + facet_wrap("trace", scales="free_x", ncol=ncol)
             + expand_limits(y=0)
             + labs(title="Multi-Trial Consistency", x="Number of Nodes",
                    y="Total Time (minutes)")
             + theme_bw()
             + theme(figure_size=(min(18, max(6, ncol * 4)), min(20, max(4, nrow * 3.5))),
                     axis_text_x=element_text(rotation=45, ha="right", size=8)))
        display(p); _maybe_save(p, "g2_multitrial")
    else:
        print("All configs have a single trial — nothing to compare.")
else:
    print("No runs to plot.")

## Graph 3 — Weak Scaling

This reads the weak-scaling curves **exactly as planned** in §2/§3 (`plan.json`'s
`weak_curves`) — there is no re-matching or tolerance applied here at analysis time.
Each planned level becomes one facet panel; a panel is skipped only if fewer than 2 of its
planned `(trace, nodes)` points have actually been collected so far. Levels built around
`WEAK_REQUIRED_TRACES` are marked **[required]** in the facet title.

**Ideal weak scaling is a flat total-time line**, equivalently a throughput line that rises
exactly proportionally with node count (dashed reference line, computed per level from its
own smallest-node point). Efficiency = t(fewest nodes) / t(nodes) is the normalized,
overlaid comparison across all levels (1.0 = perfect; flat is good).

In [ ]:
# ---- Read the weak-scaling curves that were actually PLANNED (plan.json), rather than
# re-deriving matches post-hoc -- membership was decided once, at design time in §2/§3
# (including any WEAK_REQUIRED_TRACES), so there is no tolerance/matching logic at all here.
_weak_curves_plan = {float(g): [tuple(m) for m in members]
                      for g, members in _plan.get("weak_curves", {}).items()}
_weak_required = set(_plan.get("weak_required_traces", []))

if not _weak_curves_plan:
    print("No weak-scaling curves found in plan.json for this run (COLLECTION_MODE may not "
          "have included WEAK, or this is an older run predating this format).")
else:
    frames = []
    for g, members in sorted(_weak_curves_plan.items()):
        rows = []
        for t, n in members:
            sub = df_runs[(df_runs["traced_nodes"] == t) & (df_runs["nl"] == n)]
            if sub.empty:
                continue  # not collected (yet) -- skip rather than guess
            rows.append({"traced_nodes": t, "nl": n,
                         "med": sub["totalTime"].median(), "lo": sub["totalTime"].min(),
                         "hi": sub["totalTime"].max(), "thr": sub["throughput"].median(),
                         "gpn": trace_sizes_gib[t] / n})
        if len(rows) < 2:
            continue  # need >= 2 real points to draw a line
        frame = pd.DataFrame(rows)
        frame["target"] = g
        frame["is_required"] = frame["traced_nodes"].isin(_weak_required)
        frames.append(frame)

    if not frames:
        print("No planned weak-scaling level has >= 2 collected data points yet.")
    else:
        weak_sel = pd.concat(frames, ignore_index=True)
        weak_sel["time_min"] = weak_sel["med"] / SEC_PER_MIN
        weak_sel["lo_min"] = weak_sel["lo"] / SEC_PER_MIN
        weak_sel["hi_min"] = weak_sel["hi"] / SEC_PER_MIN
        weak_sel["thr_million"] = weak_sel["thr"] / 1e6
        weak_sel["trace"] = trace_cat(weak_sel["traced_nodes"])

        curve_required = weak_sel.groupby("target")["is_required"].any()
        weak_sel["target_lab"] = weak_sel["target"].map(
            lambda g: f"~{g:.2f} GiB/node" + (" [required]" if curve_required[g] else ""))

        n_facets = weak_sel["target_lab"].nunique()
        ncol = min(4, n_facets)
        nrow = max(1, -(-n_facets // ncol))
        fig_w, fig_h = min(18, max(6, ncol * 4)), min(20, max(3.5, nrow * 3.5))

        print(f"{n_facets} weak-scaling level(s) with >= 2 collected points "
              f"({int(curve_required.sum())} include a required trace).")

        # G3a — total time per planned level (flat = ideal)
        p1 = (ggplot(weak_sel, aes("nl", "time_min"))
              + geom_ribbon(aes(ymin="lo_min", ymax="hi_min"), alpha=0.2)
              + geom_line() + geom_point(aes(color="trace"), size=2.5)
              + facet_wrap("target_lab", scales="free")
              + node_axis_linear(weak_sel["nl"]) + expand_limits(y=0)
              + scale_color_brewer(type="qual", palette="Dark2")
              + labs(title="Weak Scaling: Total Time by Planned Level (flat = ideal)",
                     x="Number of Nodes", y="Total Time (minutes)", color="Trace Size")
              + theme_bw() + theme(figure_size=(fig_w, fig_h)))
        display(p1); _maybe_save(p1, "g3a_weak_time_by_level")

        # Per-level extras: the "ideal proportional" throughput reference and efficiency,
        # each computed from that level's own smallest-node point.
        parts = []
        for _, sub in weak_sel.groupby("target"):
            sub = sub.sort_values("nl").copy()
            n_min = sub["nl"].iloc[0]
            sub["ideal_thr"] = sub["thr_million"].iloc[0] * sub["nl"] / n_min
            sub["efficiency"] = sub["med"].iloc[0] / sub["med"]
            parts.append(sub)
        weak_sel2 = pd.concat(parts, ignore_index=True)

        # G3b — throughput per level (dashed = ideal proportional growth)
        p2 = (ggplot(weak_sel2, aes("nl", "thr_million"))
              + geom_line(aes(y="ideal_thr"), color="grey", linetype="dashed")
              + geom_line() + geom_point(aes(color="trace"), size=2.5)
              + facet_wrap("target_lab", scales="free")
              + node_axis_linear(weak_sel2["nl"]) + expand_limits(y=0)
              + scale_color_brewer(type="qual", palette="Dark2")
              + labs(title="Weak Scaling: Throughput by Level (dashed = ideal proportional growth)",
                     x="Number of Nodes", y="Throughput (Million Events / s)", color="Trace Size")
              + theme_bw() + theme(figure_size=(fig_w, fig_h)))
        display(p2); _maybe_save(p2, "g3b_weak_throughput_by_level")

        # G3c — efficiency across all levels, overlaid (the one plot where overlay is the point)
        p3 = (ggplot(weak_sel2, aes("nl", "efficiency", color="target_lab"))
              + geom_hline(yintercept=1.0, linetype="dashed", color="grey")
              + geom_line() + geom_point(size=2.5)
              + node_axis_linear(weak_sel2["nl"]) + expand_limits(y=0)
              + labs(title="Weak-Scaling Efficiency by Level (1.0 = perfect; flat is good)",
                     x="Number of Nodes", y="t(Fewest Nodes) / t(Nodes)", color="Level")
              + theme_bw() + theme(figure_size=(9, 5)))
        display(p3); _maybe_save(p3, "g3c_weak_efficiency")

## Graph 4 — Per-config task profile

For every `(trace, nodes)` config: the per-locale distribution of task time (violin +
jitter), and a stacked breakdown of where each locale spends time (leaf phases only, so
nothing is double-counted). One pair of plots per config — this can be many plots.

In [ ]:
# Leaf timing columns (exclude aggregates that double-count).
LEAF_COLS = ["Open (s)", "Setup (s)", "EnterCB (s)", "LeaveCB (s)", "MetricCB (s)",
             "libOTF2 (s)", "WriteCG (s)", "WriteMet (s)"]

DISPLAY_TASKS = False  # set False to skip the per-trace task-level plots (they're slow
                       # to render in Jupyter and not needed for the main scaling graphs)
def task_profile(traced_nodes, nl):
    d = df_tasks[(df_tasks["traced_nodes"] == traced_nodes) & (df_tasks["nl"] == nl)].copy()
    if d.empty:
        return
    lbl = size_label(traced_nodes)
    width = min(14, 3 + nl * 0.4)
    d["total_min"] = d["Total (s)"] / SEC_PER_MIN
    p1 = (ggplot(d, aes("factor(Locale)", "total_min"))
          + geom_violin(fill="steelblue", alpha=0.4)
          + geom_jitter(width=0.15, alpha=0.4, size=1)
          + expand_limits(y=0)
          + labs(title=f"Per-Locale Task Time — {lbl} Trace, {nl} Nodes",
                 x="Locale ID", y="Task Total Time (minutes)")
          + theme_bw() + theme(figure_size=(width, 4)))
    if DISPLAY_TASKS:
        display(p1)

    leaves = [c for c in LEAF_COLS if c in d.columns]
    m = (d.groupby("Locale")[leaves].mean().reset_index()
           .melt("Locale", var_name="phase", value_name="seconds"))
    m["minutes"] = m["seconds"] / SEC_PER_MIN
    m["phase"] = m["phase"].str.replace(" (s)", "", regex=False)
    p2 = (ggplot(m, aes("factor(Locale)", "minutes", fill="phase"))
          + geom_col()
          + expand_limits(y=0)
          + labs(title=f"Per-Locale Timing Breakdown (Mean) — {lbl} Trace, {nl} Nodes",
                 x="Locale ID", y="Time (minutes)", fill="Phase")
          + theme_bw() + theme(figure_size=(width, 4)))
    if DISPLAY_TASKS:
        display(p2)


if not df_tasks.empty:
    cfgs = (df_tasks[["traced_nodes", "nl"]].drop_duplicates()
            .sort_values(["traced_nodes", "nl"]))
    print(f"{len(cfgs)} config(s) with task-level data.")
    for _, c in cfgs.iterrows():
        task_profile(int(c["traced_nodes"]), int(c["nl"]))
else:
    print("No task-level data.")

## Graph 5 — Phase Breakdown (and Saving Figures)

Mean wall time per phase for each config, faceted by trace size (the `Total` row is
excluded) so the axis stays readable regardless of how many node counts were collected. To
export every figure as PNG, set `SAVE_PLOTS = True` in the analysis-config cell and re-run
the graph cells — each one calls `_maybe_save(...)` into the run's `plots/` folder.

In [ ]:
if not df_phases.empty:
    d = df_phases[df_phases["phase"] != "Total"].copy()
    g = d.groupby(["traced_nodes", "nl", "phase"])["time"].mean().reset_index()
    g["minutes"] = g["time"] / SEC_PER_MIN
    g["trace"] = trace_cat(g["traced_nodes"])
    n_facets = g["trace"].nunique()
    ncol = min(4, n_facets) if n_facets else 1
    nrow = max(1, -(-n_facets // ncol)) if ncol else 1
    p = (ggplot(g, aes("factor(nl)", "minutes", fill="phase"))
         + geom_col()
         + facet_wrap("trace", scales="free_x", ncol=ncol)
         + expand_limits(y=0)
         + labs(title="Phase Breakdown (Mean Across Trials)",
                x="Number of Nodes", y="Time (minutes)", fill="Phase")
         + theme_bw()
         + theme(figure_size=(min(18, max(6, ncol * 4)), min(20, max(4, nrow * 3.5))),
                 axis_text_x=element_text(rotation=45, ha="right", size=8)))
    display(p)
else:
    print("No phase-level data.")